# NB_hospital_daily_summary

Builds `adb_mediflow.gold.hospital_daily_summary` (one row per hospital per day) from silver lab results, hospitals and calendar, with a day-over-day test volume trend, per the Silver to Gold STTM. Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_mediflow.gold.hospital_daily_summary AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                            AS summary_date,
    count(*)                             AS total_tests,
    count(DISTINCT patient_id)           AS unique_patients_tested,
    count(DISTINCT doctor_id)            AS active_doctor_count,
    count_if(result_status = 'Abnormal') AS abnormal_test_count,
    count_if(result_status = 'Critical') AS critical_test_count
  FROM adb_mediflow.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  d.hospital_id,
  h.hospital_name,
  d.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(d.total_tests AS INT)            AS total_tests,
  CAST(d.unique_patients_tested AS INT) AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)    AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)    AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)    AS critical_test_count,
  CAST(d.abnormal_test_count / d.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(d.critical_test_count / d.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)   AS prev_day_total_tests,
  CAST(CASE WHEN d.prev_day_total_tests IS NULL OR d.prev_day_total_tests = 0 THEN NULL
            ELSE (d.total_tests - d.prev_day_total_tests) / d.prev_day_total_tests * 100 END AS DECIMAL(6,2)) AS test_volume_change_pct
FROM with_lag d
LEFT JOIN adb_mediflow.silver.hospitals h ON d.hospital_id = h.hospital_id
LEFT JOIN adb_mediflow.silver.calendar c  ON c.date_key = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT);

In [0]:
# %sql
# SELECT count(*) AS row_count, count(DISTINCT hospital_id, summary_date) AS distinct_grain,
#        sum(total_tests) AS total_tests, count_if(day_name IS NULL) AS missing_calendar,
#        count_if(prev_day_total_tests IS NULL) AS first_day_rows, count(DISTINCT hospital_id) AS hospitals
# FROM adb_mediflow.gold.hospital_daily_summary;

In [0]:
updated_df = spark.table("adb_mediflow.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))